# Path Sampling

Path sampling runs many A* searches with varied dropout to generate diverse paths between two crystal structures, then evaluates their energies using a machine learning potential. The path with the lowest barrier represents the best estimate of the minimum energy pathway found so far.

This guide explains how path sampling works and how to use it.

## The Role of Dropout in Path Diversity

A* search is deterministic: given the same inputs, it always produces the same path. This is problematic when we want to explore different regions of the energy landscape. The path A* finds might have a high barrier, while a slightly different path could have a much lower one.

Dropout introduces controlled randomness. At each step of A*, some fraction of valid neighbors are randomly excluded from consideration. Different dropout values (and different random number sequences) lead A* to explore different parts of the search space and find different paths.

By running many searches with varied dropout, we sample the space of possible paths. The more samples we take, the more likely we are to find a good (low barrier) path.

## Energy Evaluation with GRACE

Each path found by A* is just a sequence of CNF coordinates. To measure its barrier, we need to evaluate the energy of each structure along the path. The barrier is simply the maximum energy encountered.

We use GRACE, a machine learning interatomic potential, to evaluate energies. GRACE provides fast, reasonably accurate energies without running expensive DFT calculations. The energies are not as accurate as DFT, but they're good enough to identify promising paths that can be refined later.

Energy evaluation is often the computational bottleneck. Each path might contain hundreds of structures, and each energy evaluation takes a few milliseconds. To avoid redundant calculations, we maintain an energy cache: if the same CNF coordinate appears in multiple paths, we reuse the cached energy rather than recomputing it.

## A Simple Example

Let's sample paths between two zirconium structures. We'll first convert the structures to CNF, then run path sampling.

In [ ]:
from pymatgen.core import Structure, Lattice
from cnf import CNFConstructor

# HCP zirconium (2 atoms per cell)
a, c = 3.23, 5.15
lattice_start = Lattice.hexagonal(a, c)
zr_start = Structure(
    lattice_start,
    ["Zr", "Zr"],
    [[0, 0, 0], [1/3, 2/3, 0.5]]
)

# Strained version (5% larger a parameter)
lattice_goal = Lattice.hexagonal(a * 1.05, c)
zr_goal = Structure(
    lattice_goal,
    ["Zr", "Zr"],
    [[0, 0, 0], [1/3, 2/3, 0.5]]
)

# Convert to CNF at moderate resolution
xi = 1.0
delta = 50
constructor = CNFConstructor(xi=xi, delta=delta)

start_cnf = constructor.from_pymatgen_structure(zr_start).cnf
goal_cnf = constructor.from_pymatgen_structure(zr_goal).cnf

print(f"Start CNF: {start_cnf.coords[:7]}...")
print(f"Goal CNF:  {goal_cnf.coords[:7]}...")

In [ ]:
from cnf.navigation.astar.iterative import sample

result = sample(
    start_cnfs=[start_cnf],
    goal_cnfs=[goal_cnf],
    num_samples=10,
    dropout_range=(0.05, 0.15),
    min_distance=1.5,
    max_iterations=5000,
    verbosity=1,
)

The output shows each attempt with its dropout value, whether a path was found, and if so, the path length and barrier height. At the end, we see summary statistics including the best barrier found.

In [ ]:
print(f"Paths found: {len(result.paths)}/{len(result.attempts)}")
print(f"Best barrier: {result.best_barrier:.4f} eV")
print(f"Best path length: {len(result.best_path)} steps")

## Understanding the Result

The `SearchResult` contains all attempts, both successful and unsuccessful. Each attempt records whether a path was found, how many A* iterations it took, and the dropout value used.

In [ ]:
# Look at individual attempts
for i, attempt in enumerate(result.attempts[:5]):
    if attempt.found:
        print(f"Attempt {i+1}: dropout={attempt.metadata['dropout']:.2f}, "
              f"barrier={attempt.path.barrier:.4f} eV, "
              f"length={len(attempt.path)}")
    else:
        print(f"Attempt {i+1}: dropout={attempt.metadata['dropout']:.2f}, no path found")

## Examining the Best Path

The best path is the one with the lowest barrier. We can examine its energy profile to understand the transformation.

In [ ]:
best_path = result.best_path

print(f"Path length: {len(best_path)} steps")
print(f"Barrier: {best_path.barrier:.4f} eV")
print()
print("Energy profile (every 10th point):")
for i in range(0, len(best_path.energies), 10):
    print(f"  Step {i:3d}: {best_path.energies[i]:.4f} eV")

In [ ]:
# What types of steps does the path contain?
step_types = best_path.get_step_types()
if step_types:
    lattice_steps = step_types.count('lattice')
    motif_steps = step_types.count('motif')
    print(f"Lattice steps: {lattice_steps}")
    print(f"Motif steps: {motif_steps}")

## Dropout Range

The `dropout_range` parameter controls the range of dropout values used across attempts. Each attempt samples uniformly from this range.

Low dropout (0.02-0.05) produces paths similar to the deterministic A* result, with small variations. High dropout (0.2-0.3) produces very different paths but may fail to find any path at all because too many neighbors are excluded.

A moderate range like (0.05, 0.1) balances path diversity with success rate. If you find that many attempts fail, try lowering the upper bound. If all paths look similar, try raising the lower bound.

In [ ]:
# Try with higher dropout for more diversity
result_high_dropout = sample(
    start_cnfs=[start_cnf],
    goal_cnfs=[goal_cnf],
    num_samples=10,
    dropout_range=(0.1, 0.2),
    min_distance=1.5,
    max_iterations=5000,
    verbosity=1,
)

## Bidirectional Sampling

By default, all sampling attempts search from start to goal. The `bidirectional` option causes about half the attempts to search in the reverse direction (goal to start). This can help discover paths that are easier to find in one direction than the other.

Note that when a reverse path is found, it's automatically reversed before energy evaluation, so all paths in the result go from start to goal regardless of which direction was searched.

In [ ]:
# Bidirectional sampling
result_bidir = sample(
    start_cnfs=[start_cnf],
    goal_cnfs=[goal_cnf],
    num_samples=10,
    dropout_range=(0.05, 0.1),
    bidirectional=True,
    min_distance=1.5,
    max_iterations=5000,
    verbosity=1,
)

# Check the directions used
forward = sum(1 for a in result_bidir.attempts if a.metadata.get('direction') == 'forward')
backward = sum(1 for a in result_bidir.attempts if a.metadata.get('direction') == 'backward')
print(f"\nForward attempts: {forward}, Backward attempts: {backward}")

## Parallel Sampling

Sampling attempts are independent and can be run in parallel. The `n_workers` parameter controls parallelization. Each worker creates its own energy calculator, so GPU memory and CPU resources are distributed across workers.

The implementation uses Python's ProcessPoolExecutor with the 'spawn' context to avoid issues with TensorFlow and fork. Each worker receives a `CalcProvider` (a picklable factory) and creates its own calculator instance, with TensorFlow threading adjusted to avoid contention.

In [ ]:
# Parallel sampling with 4 workers
result_parallel = sample(
    start_cnfs=[start_cnf],
    goal_cnfs=[goal_cnf],
    num_samples=20,
    dropout_range=(0.05, 0.1),
    min_distance=1.5,
    max_iterations=5000,
    n_workers=4,
    verbosity=1,
)

## Custom Calculator

By default, sampling uses a GRACE foundation model. If you have a fine-tuned model for your material system, you can provide a custom calculator via `CalcProvider`.

The `CalcProvider` is a picklable factory that creates calculator instances. This pattern is necessary because calculator objects (which contain loaded ML models) cannot be directly pickled and sent to worker processes. Instead, each worker receives the provider and creates its own calculator.

In [ ]:
from cnf.calculation.grace import GraceCalcProvider

# Using a custom model path (commented out since path is hypothetical)
# custom_provider = GraceCalcProvider(model_path="/path/to/fine_tuned_model")
# result_custom = sample(
#     start_cnfs=[start_cnf],
#     goal_cnfs=[goal_cnf],
#     calc_provider=custom_provider,
#     num_samples=10,
#     max_iterations=5000,
# )

# For testing with constant energy (no ML model needed)
from cnf.calculation.constant_calculator import ConstantCalcProvider

test_provider = ConstantCalcProvider(val=0.0)
result_test = sample(
    start_cnfs=[start_cnf],
    goal_cnfs=[goal_cnf],
    calc_provider=test_provider,
    num_samples=5,
    dropout_range=(0.05, 0.1),
    min_distance=1.5,
    max_iterations=5000,
    verbosity=1,
)

print(f"\nWith constant energy, all barriers are: {result_test.best_barrier}")

## Saving and Loading Results

Sampling results can be saved to JSON. The `output_dir` parameter causes automatic saving, with intermediate saves every 10 attempts (useful for long runs that might be interrupted).

In [ ]:
import tempfile
import os

with tempfile.TemporaryDirectory() as tmpdir:
    result_saved = sample(
        start_cnfs=[start_cnf],
        goal_cnfs=[goal_cnf],
        num_samples=10,
        dropout_range=(0.05, 0.1),
        min_distance=1.5,
        max_iterations=5000,
        output_dir=tmpdir,
        verbosity=0,
    )
    
    saved_path = os.path.join(tmpdir, "sample_result.json")
    print(f"Saved to: {saved_path}")
    print(f"File exists: {os.path.exists(saved_path)}")

In [ ]:
# Loading a saved result
from cnf.navigation.astar.models import SearchResult

with tempfile.TemporaryDirectory() as tmpdir:
    outpath = os.path.join(tmpdir, "sample_result.json")
    result.to_json(outpath)
    
    loaded = SearchResult.from_json(outpath)
    print(f"Loaded {len(loaded.paths)} paths")
    print(f"Best barrier: {loaded.best_barrier:.4f} eV")

## Using the Results

The main output of path sampling is the best barrier found across all sampled paths. This gives an upper bound on the true minimum energy barrier between the two structures. More samples generally lead to better (lower) estimates.

The sampling result also provides useful statistics like the maximum iterations used by successful attempts, which can help calibrate iteration limits for future searches.

In [ ]:
print(f"Best barrier: {result.best_barrier:.4f} eV")
print(f"Max successful iterations: {result.max_successful_iterations}")
print(f"Success rate: {result.success_rate:.1%}")

## Summary

Path sampling generates diverse transformation pathways by running many A* searches with varied dropout. Each path is evaluated using a machine learning potential to measure its energy barrier.

The main outputs are:

1. Multiple paths with their energies and barriers
2. The best (lowest) barrier found
3. Iteration statistics that can help calibrate future searches

The best barrier found in sampling is an upper bound on the true minimum barrier. Taking more samples increases the likelihood of finding lower-barrier paths.